# Lab 03: LangChain Integration

**Goal:** Use CallbackHandler to trace LangChain chains and agents with LangFuse.

**What you'll learn:**
- How to integrate LangFuse with LangChain using CallbackHandler
- How to trace RAG pipelines end-to-end
- Key handler configuration parameters for production use
- How to instrument chains with a single line of code

In [ ]:
import os
import shutil
import textwrap

WORKDIR = "/tmp/ailab-11-03"

if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
os.makedirs(WORKDIR, exist_ok=True)

## Step 1: CallbackHandler Pattern

In [ ]:
print("One-line integration with LangChain:\n")

code_example = textwrap.dedent("""\
    from langfuse.callback import CallbackHandler

    handler = CallbackHandler(
        public_key="pk-lf-...",
        secret_key="sk-lf-...",
        host="http://localhost:3000",
        user_id="alice",
        session_id="chat_42",
        tags=["production", "v2.0"],
    )

    # Pass handler to any LangChain invoke
    result = chain.invoke(
        {"question": "What is RAG?"},
        config={"callbacks": [handler]},
    )
""")

for line in code_example.strip().split("\n"):
    print(f"    {line}")

In [ ]:
print("What gets captured automatically:")
captured = [
    ("LLM calls",      "Model, input, output, tokens, cost, latency"),
    ("Tool calls",      "Tool name, input, output, duration"),
    ("Chain execution", "Each chain step with input/output"),
    ("Retriever calls", "Query, retrieved documents, scores"),
]
for what, detail in captured:
    print(f"    {what:<18} {detail}")

## Step 2: RAG Pipeline Tracing

In [ ]:
rag_example = textwrap.dedent("""\
    from langchain.chains import RetrievalQA
    from langchain_community.vectorstores import Chroma

    # Set up retriever
    vectorstore = Chroma(persist_directory="./chroma_db")
    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

    # Create chain
    chain = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=retriever,
        chain_type="stuff",
    )

    # Invoke with LangFuse callback
    result = chain.invoke(
        {"query": "How does RAG work?"},
        config={"callbacks": [handler]},
    )
""")

print("RAG pipeline with LangFuse tracing:\n")
for line in rag_example.strip().split("\n"):
    print(f"    {line}")

In [ ]:
print("Trace structure for RAG:")
print("    Trace: chat_request")
print("    \u251c\u2500\u2500 Span: retriever")
print("    \u2502   \u2514\u2500\u2500 (query: 'How does RAG work?', docs: 5)")
print("    \u2514\u2500\u2500 Span: llm_chain")
print("        \u2514\u2500\u2500 Generation: groq/llama3-70b")
print("            \u251c\u2500\u2500 input_tokens: 1,200")
print("            \u251c\u2500\u2500 output_tokens: 350")
print("            \u2514\u2500\u2500 cost: $0.0062")

## Step 3: Handler Configuration

In [ ]:
print("Key parameters for CallbackHandler:\n")
params = [
    ("user_id",     "Identifies the user",        "Required for cost-per-user tracking"),
    ("session_id",  "Groups related requests",     "Conversation thread ID"),
    ("tags",        "Categorize traces",           '["production", "v2.0", "experiment"]'),
    ("metadata",    "Custom key-value pairs",      '{"endpoint": "/chat", "model": "llama3"}'),
    ("trace_name",  "Override default trace name",  '"rag_query" or "agent_run"'),
]
print(f"    {'Param':<14} {'Purpose':<30} {'Example'}")
print(f"    {'-'*80}")
for param, purpose, example in params:
    print(f"    {param:<14} {purpose:<30} {example}")

## TODO 1: LangChain Instrumentation Code

Write code that:
- Imports CallbackHandler from langfuse.callback
- Creates handler with user_id, session_id, tags
- Creates a RetrievalQA chain with retriever
- Invokes chain with callbacks=[handler]
- Gets trace_id from handler

In [ ]:
# TODO: Instrument a LangChain RAG pipeline with LangFuse
# Include: import, handler creation, chain invocation

todo1_code = textwrap.dedent("""\
    # TODO: Instrument a LangChain RAG pipeline with LangFuse
    # Include: import, handler creation, chain invocation

""")

with open(os.path.join(WORKDIR, "instrumented_rag.py"), "w") as f:
    f.write(todo1_code)

In [ ]:
checks1 = [
    ("Has CallbackHandler import",  "CallbackHandler" in todo1_code),
    ("Has langfuse import",         "langfuse" in todo1_code),
    ("Has handler creation",        "CallbackHandler(" in todo1_code),
    ("Has user_id",                 "user_id" in todo1_code),
    ("Has session_id",              "session_id" in todo1_code),
    ("Has tags",                    "tags" in todo1_code),
    ("Has chain invoke",            "invoke" in todo1_code),
    ("Has callbacks config",        "callbacks" in todo1_code),
    ("Has handler reference",       "handler" in todo1_code),
    ("Has trace_id retrieval",      "trace_id" in todo1_code or "get_trace" in todo1_code),
]

score1 = sum(1 for _, ok in checks1 if ok)
print(f"Validating ({score1}/{len(checks1)}):\n")
for name, ok in checks1:
    print(f"    [{'PASS' if ok else 'FAIL'}] {name}")

## TODO 2: Integration Quiz

Fill in the answers for each question below.

In [ ]:
quiz = [
    {
        "question": "What LangFuse class integrates with LangChain?",
        "answer": "___",
        "correct": "callbackhandler",
    },
    {
        "question": "What config key passes the handler to chain.invoke()?",
        "answer": "___",
        "correct": "callbacks",
    },
    {
        "question": "What handler parameter groups requests into conversations?",
        "answer": "___",
        "correct": "session_id",
    },
    {
        "question": "What trace level captures the actual LLM API call?",
        "answer": "___",
        "correct": "generation",
    },
]

# YOUR CODE HERE: Fill in quiz answers
# quiz[0]["answer"] = "CallbackHandler"
# quiz[1]["answer"] = ???
# ...

In [ ]:
score2 = 0
for i, q in enumerate(quiz, 1):
    answer = q["answer"].strip().lower().replace("_", "").replace(" ", "")
    correct = q["correct"].lower().replace("_", "").replace(" ", "")
    is_correct = answer == correct

    if q["answer"] == "___":
        status = "TODO"
    elif is_correct:
        status = "PASS"
        score2 += 1
    else:
        status = "FAIL"
    print(f"    [{status}] Q{i}: {q['question']}")

print(f"\n  Score: {score2}/{len(quiz)}")

## Summary

In [ ]:
print("Key concepts:")
print("    1. CallbackHandler = one-line LangChain integration")
print("    2. Pass handler via config={'callbacks': [handler]}")
print("    3. Automatically captures LLM calls, tools, chains, retrievers")
print("    4. Set user_id + session_id for per-user tracking")
print(f"\n  TODO 1: {score1}/{len(checks1)} instrumentation checks")
print(f"  TODO 2: {score2}/{len(quiz)} quiz answers correct")
print(f"\n  Files generated in {WORKDIR}/")

## Key Takeaways

- **CallbackHandler** provides one-line integration between LangChain and LangFuse
- Pass the handler via `config={"callbacks": [handler]}` to any LangChain `.invoke()` call
- LangFuse automatically captures LLM calls, tool calls, chain execution, and retriever calls
- Configure `user_id` and `session_id` on the handler for per-user cost tracking and conversation grouping
- Use `tags` and `metadata` to categorize and enrich traces for filtering in the LangFuse dashboard